要运行此程序，请在**免费** Tesla T4 Google Colab 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

# 目标：让 Gemma 4 通过强化学习解决数独难题

我们的目标是让 Gemma 4 学会使用强化学习 (GRPO) 解决数独难题。
该模型将设计一种策略来填充空单元格，我们将奖励它正确的放置位置
并完成有效的谜题。

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Sudoku_Puzzle_by_L2G-20050714_solution_standardized_layout.svg/1280px-Sudoku_Puzzle_by_L2G-20050714_solution_standardized_layout.svg.png"高度=“300”/>

# 安装
我们将使用 [Unsloth](https://github.com/unslothai/unsloth) 在 Gemma 4 上进行强化学习。Unsloth 节省了 70% 的 VRAM 使用量，并使强化学习速度提高 2 到 6 倍。

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    # Gemma 4 需要 Transformer >= 5.5.0 — 不要在此处固定到 4.x
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356# 子目录=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
# Gemma 4 需要变形金刚 >= 5.5.0
!uv pip install --upgrade --no-deps "transformers>=5.5.0" "tokenizers>=0.22.0,<=0.23.0" "trl>=0.28.0" unsloth unsloth_zoo

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # 适用于 Gemma 4 视觉/音频

### 不懒惰

In [ ]:
from unsloth import FastVisionModel
import torch
max_seq_length = 4096 # 可以增加更长的推理痕迹
lora_rank = 32 # 排名越高=越聪明，但速度越慢

gemma4_models = [
    # Gemma-4 指令模型：
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 基本型号：
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # 更多模型请访问 https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # LoRA 16 位错误
    fast_inference = False, # 启用 vllm 快速推理
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

为了进行高效的强化学习，我们将使用 [LoRA](https://arxiv.org/abs/2106.09685)，它允许我们仅向模型添加 1 到 5% 的额外权重以进行微调。这使我们能够节省 60% 以上的内存使用量，同时保持良好的准确性。

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    r = lora_rank, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 加快训练速度
    use_gradient_checkpointing = "unsloth", # 减少内存使用
    random_state = 3407,
)

# 数独游戏实现

我们使用 GPT-5 创建一个干净的数独求解器环境。该策略输出“row,col,value”来填充单元格。

In [ ]:
# @title 数独游戏实现
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import random
import copy

def _is_valid_placement(board: List[List[int]], row: int, col: int, num: int) -> bool:
    """Check if placing num at (row, col) is valid."""
    # 检查行
    if num in board[row]:
        return False

    # 检查栏
    if num in [board[r][col] for r in range(9)]:
        return False

    # 检查 3x3 框
    box_row, box_col = 3 * (row // 3), 3 * (col // 3)
    for r in range(box_row, box_row + 3):
        for c in range(box_col, box_col + 3):
            if board[r][c] == num:
                return False

    return True

def _solve_sudoku(board: List[List[int]]) -> bool:
    """Solve sudoku using backtracking (for puzzle generation)."""
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                for num in range(1, 10):
                    if _is_valid_placement(board, row, col, num):
                        board[row][col] = num
                        if _solve_sudoku(board):
                            return True
                        board[row][col] = 0
                return False
    return True

def _generate_complete_board(rng: random.Random) -> List[List[int]]:
    """Generate a complete valid Sudoku board."""
    board = [[0 for _ in range(9)] for _ in range(9)]

    # 先填充对角线 3x3 框（它们不会互相影响）
    for box in range(3):
        nums = list(range(1, 10))
        rng.shuffle(nums)
        for i in range(3):
            for j in range(3):
                board[box * 3 + i][box * 3 + j] = nums[i * 3 + j]

    # 解决剩下的问题
    _solve_sudoku(board)
    return board

@dataclass
class SudokuGame:
    difficulty: int = 40  # 要去除的细胞数量（20 = 简单，40 = 中等，50 = 困难）
    seed: Optional[int] = None
    _rng: random.Random = field(init = False, repr = False)
    _board: List[List[int]] = field(init = False, repr = False)
    _solution: List[List[int]] = field(init = False, repr = False)
    _initial_board: List[List[int]] = field(init = False, repr = False)
    _moves: int = field(default = 0, init = False, repr = False)
    _state: str = field(default = "ongoing", init = False, repr = False)

    def __post_init__(self):
        self._rng = random.Random(self.seed)

        # 生成完整的板子
        complete_board = _generate_complete_board(self._rng)
        self._solution = copy.deepcopy(complete_board)

        # 删除细胞以创建拼图
        self._board = copy.deepcopy(complete_board)
        cells = [(r, c) for r in range(9) for c in range(9)]
        self._rng.shuffle(cells)

        for r, c in cells[:self.difficulty]:
            self._board[r][c] = 0

        self._initial_board = copy.deepcopy(self._board)
        self._update_state()

    def board(self) -> List[List[int]]:
        """Return current board state."""
        return [row[:] for row in self._board]

    def initial_board(self) -> List[List[int]]:
        """Return initial puzzle state."""
        return [row[:] for row in self._initial_board]

    def state(self) -> str:
        """Return game state: 'ongoing', 'success', or 'failed'."""
        return self._state

    def moves(self) -> int:
        """Return number of moves made."""
        return self._moves

    def place_number(self, row: int, col: int, num: int) -> bool:
        """Place a number on the board. Returns True if valid move."""
        # 验证输入
        if not (0 <= row < 9 and 0 <= col < 9):
            self._state = "failed"
            return False

        if not (1 <= num <= 9):
            self._state = "failed"
            return False

        # 无法修改初始单元格
        if self._initial_board[row][col] != 0:
            self._state = "failed"
            return False
        if self._board[row][col] != 0:
            self._state = "failed"
            return False
        # 检查放置是否有效
        if not _is_valid_placement(self._board, row, col, num):
            self._state = "failed"
            return False

        # 名额
        self._board[row][col] = num
        self._moves += 1
        self._update_state()
        return True

    def _update_state(self) -> None:
        """Update game state based on current board."""
        # 检查拼图是否完整
        if all(self._board[r][c] != 0 for r in range(9) for c in range(9)):
            # 验证解决方案是否正确
            if self._board == self._solution:
                self._state = "success"
            else:
                self._state = "failed"
        else:
            self._state = "ongoing"

    def pretty(self, colors: bool = True) -> str:
        """Pretty print the Sudoku board."""
        RESET = "\x1b[0m"
        INITIAL = "\x1b[38;5;45m"   # 青色表示初始数字
        PLACED = "\x1b[38;5;226m"    # 黄色表示已放置的号码
        EMPTY = "\x1b[38;5;239m"     # 灰色表示空单元格

        lines = []
        lines.append("┌───────┬───────┬───────┐")

        for row in range(9):
            row_str = "│ "
            for col in range(9):
                num = self._board[row][col]

                if colors:
                    if num == 0:
                        row_str += f"{EMPTY}.{RESET}"
                    elif self._initial_board[row][col] != 0:
                        row_str += f"{INITIAL}{num}{RESET}"
                    else:
                        row_str += f"{PLACED}{num}{RESET}"
                else:
                    row_str += str(num) if num != 0 else "."

                if col % 3 == 2:
                    row_str += " │ "
                else:
                    row_str += " "

            lines.append(row_str.rstrip())

            if row == 8:
                lines.append("└───────┴───────┴───────┘")
            elif row % 3 == 2:
                lines.append("├───────┼───────┼───────┤")

        return "\n".join(lines)

测试数独环境：

In [ ]:
# 创建一个简单的谜题
game = SudokuGame(difficulty = 30, seed = 42)
print("初始谜题：")
print(game.pretty())
print(f"\nState: {game.state()}, Moves: {game.moves()}")

Initial puzzle:
┌───────┬───────┬───────┐
│ 4 . 8 │ . 5 2 │ 6 3 . │
│ . 9 3 │ 4 6 7 │ . . 1 │
│ 6 1 2 │ . 9 8 │ 4 . . │
├───────┼───────┼───────┤
│ 1 . 4 │ . . . │ 7 9 5 │
│ 3 . 9 │ 7 1 . │ 8 2 6 │
│ 7 8 . │ 5 . 9 │ 1 . 3 │
├───────┼───────┼───────┤
│ 2 4 . │ 9 7 . │ . 6 . │
│ 8 3 5 │ 6 4 . │ . 7 . │
│ . . 7 │ 2 . . │ . 1 4 │
└───────┴───────┴───────┘

State: ongoing, Moves: 0


In [ ]:
game

SudokuGame(difficulty=30, seed=42)

尝试做一些动作：

In [ ]:
# 做出有效的举动
game.place_number(0, 1, 7)
print("\n将 7 放在 (1,0) 后：")
print(game.pretty())
print(f"State: {game.state()}, Moves: {game.moves()}")


After placing 7 at (1,0):
┌───────┬───────┬───────┐
│ 4 7 8 │ . 5 2 │ 6 3 . │
│ . 9 3 │ 4 6 7 │ . . 1 │
│ 6 1 2 │ . 9 8 │ 4 . . │
├───────┼───────┼───────┤
│ 1 . 4 │ . . . │ 7 9 5 │
│ 3 . 9 │ 7 1 . │ 8 2 6 │
│ 7 8 . │ 5 . 9 │ 1 . 3 │
├───────┼───────┼───────┤
│ 2 4 . │ 9 7 . │ . 6 . │
│ 8 3 5 │ 6 4 . │ . 7 . │
│ . . 7 │ 2 . . │ . 1 4 │
└───────┴───────┴───────┘
State: ongoing, Moves: 1


如果我们执行一些不属于动作空间的其他动作，我们将收到错误，并且游戏将不再接受任何动作。

# 强化学习环境设置

执行有时间限制的策略以防止无限循环。

In [ ]:
from typing import Callable
from unsloth import execute_with_time_limit

def _execute_strategy(strategy: Callable, game: SudokuGame):
    """Execute a strategy function on a Sudoku game."""
    assert callable(strategy)

    max_moves = 100
    valid_moves = 0  # 追踪成功的举动

    while game.state() == "ongoing" and valid_moves < max_moves:
        try:
            board = game.board()
            initial = game.initial_board()
            result = strategy(board, initial)

            # 验证结果格式
            if not isinstance(result, (tuple, list)) or len(result) != 3:
                # 格式无效=立即失败，但返回有效的移动
                return valid_moves, "failed"

            row, col, num = result

            # 验证类型
            if not all(isinstance(x, int) for x in [row, col, num]):
                return valid_moves, "failed"

            # 尝试放置号码
            success = game.place_number(row, col, num)

            if success:
                valid_moves += 1  # 计算本次有效移动
            else:
                # 无效移动 = 游戏失败，但返回迄今为止进行的 valid_moves
                return valid_moves, "failed"

        except Exception:
            return valid_moves, "failed"

    if valid_moves >= max_moves and game.state() == "ongoing":
        return valid_moves, "failed"

    return valid_moves, game.state()

为了允许更长的强化学习策略，我们将允许 10 秒的计时器。

In [ ]:
@execute_with_time_limit(10)
def execute_strategy(strategy: Callable, game: SudokuGame):
    """Execute strategy with 10 second time limit."""
    return _execute_strategy(strategy, game)

使用简单的策略进行测试：

In [ ]:
def simple_strategy(board, initial):
    """Simple strategy: fill first empty cell with 1."""
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0 and initial[r][c] == 0:
                return (r, c, 7)
    return (0, 0, 7)

game = SudokuGame(difficulty = 30, seed = 42)
try:
    moves, state = execute_strategy(simple_strategy, game)
    print(f"Moves: {moves}, State: {state}")
except TimeoutError as e:
    print(f"Timed out: {e}")

Moves: 1, State: failed


In [ ]:
print(game.pretty())

┌───────┬───────┬───────┐
│ 4 7 8 │ . 5 2 │ 6 3 . │
│ . 9 3 │ 4 6 7 │ . . 1 │
│ 6 1 2 │ . 9 8 │ 4 . . │
├───────┼───────┼───────┤
│ 1 . 4 │ . . . │ 7 9 5 │
│ 3 . 9 │ 7 1 . │ 8 2 6 │
│ 7 8 . │ 5 . 9 │ 1 . 3 │
├───────┼───────┼───────┤
│ 2 4 . │ 9 7 . │ . 6 . │
│ 8 3 5 │ 6 4 . │ . 7 . │
│ . . 7 │ 2 . . │ . 1 4 │
└───────┴───────┴───────┘


# 代码执行

要执行并创建一个新的Python函数，我们首先必须检查该函数是否没有调用其他全局变量或作弊。这称为“对抗奖励黑客”，因为我们不希望该函数作弊。

例如，下面的代码就很好，因为它只导入 Python 级别的函数。我们使用“check_python_modules”：

In [ ]:
from unsloth import check_python_modules, create_locked_down_function

# 测试安全代码
sample = """
def strategy(board, initial):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                return (r, c, 1)
    return (0, 0, 1)
"""

ok, info = check_python_modules(sample)
print("安全的Python代码？", ok)
print(info)

Safe Python code? True
{'stdlib': [], 'non_stdlib': [], 'relative_imports': 0}


对于下面的代码，由于我们导入了“numpy”，所以我们不应该允许执行：

In [ ]:
sample = """
def strategy(board, initial):
    import numpy as np
    return (0, 0, 1)
"""

ok, info = check_python_modules(sample)
print("安全的Python代码？", ok)
print(info)

Safe Python code? False
{'stdlib': [], 'non_stdlib': ['numpy'], 'relative_imports': 0}


# 数据和强化学习任务设置

创建指示模型生成数独求解策略的提示。您可以将其自定义为其他 RL 任务的其他任务。

In [ ]:
prompt = """
Create a Sudoku solving strategy using only native Python built-in functions without any import statements.
You are given two lists of lists (9x9 grids):
- board: current state (0 means empty)
- initial: starting puzzle (0 means was empty, numbers are fixed)

Return a tuple (row, col, number) for the next move.
- row: 0-8 (row index)
- col: 0-8 (column index)
- number: 1-9 (digit to place)

Only place numbers in cells that are BOTH empty in initial AND empty in board (initial[row][col] == 0 AND board[row][col] == 0)
Use Sudoku rules: no duplicates in rows, columns, or 3x3 boxes.
Output your function in backticks:
```python
def strategy(board, initial):
    # 你的逻辑在这里
    return (row, col, number)
```
All helper functions must be inside def strategy. Output only the function.
""".strip()

print(prompt)

Create a Sudoku solving strategy using only native Python built-in functions without any import statements.
You are given two lists of lists (9x9 grids):
- board: current state (0 means empty)
- initial: starting puzzle (0 means was empty, numbers are fixed)

Return a tuple (row, col, number) for the next move.
- row: 0-8 (row index)
- col: 0-8 (column index)
- number: 1-9 (digit to place)

Only place numbers in cells that are BOTH empty in initial AND empty in board (initial[row][col] == 0 AND board[row][col] == 0)
Use Sudoku rules: no duplicates in rows, columns, or 3x3 boxes.
Output your function in backticks:
```python
def strategy(board, initial):
    # Your logic here
    return (row, col, number)
```
All helper functions must be inside def strategy. Output only the function.


首先，让我们在没有 RL 的情况下提示模型，看看效果如何：

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt.strip()}],
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
print("=" * 50)
print("基本模型输出（强化学习训练之前）：")
print("=" * 50)

inputs = tokenizer(
    text = text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

BASE MODEL OUTPUT (before RL training):
This is a complex request. Implementing a full, robust Sudoku solver strategy using *only* native Python built-in functions (no imports like `collections.Counter` or complex data structures beyond standard lists/dicts) requires implementing the core logic of constraint checking and candidate generation.

Since the goal is to find *a* valid next move, we will use a simple backtracking/constraint propagation approach:
1. Identify all empty cells.
2. For each empty cell, determine the set of valid numbers (1-9) that can be placed there without violating Sudoku rules based on the current `board`.
3.


# 奖励功能

我们现在设计一个“extract_function”函数，它简单地提取包含在 3 个反引号中的函数。

以及 3 个奖励函数：

1. `function_works` 如果策略是有效的 Python 函数，则会奖励模型。
2. `no_cheating` 检查函数是否导入了其他模块，如果导入了，我们就会对其进行惩罚。
3. `strategy_succeeds` 在运行自动生成的策略后检查游戏策略是否确实成功获得数独。

In [ ]:
def extract_function(text):
    """Extract Python function from markdown code blocks."""
    if text.count("```") >= 2:
        first = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first:second].strip()
        fx = fx.removeprefix("python\n")
        fx = fx[fx.find("def"):]
        if fx.startswith("def strategy(board, initial):"):
            return fx
    return None

**奖励 1：功能有效**

检查生成的代码是否是有效的 Python 并且可以执行。

In [ ]:
def function_works(completions, **kwargs):
    """Reward for generating valid executable Python code."""
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)

        if function is not None:
            ok, info = check_python_modules(function)

        if function is None or "error" in info:
            score = -2.0  # 无效功能
        else:
            try:
                new_strategy = create_locked_down_function(function)
                score = 1.0  # 有效功能
            except:
                score = -1.0  # 函数有错误

        scores.append(score)
    return scores

**奖励2：不作弊**

惩罚导入外部库的函数。

In [ ]:
def no_cheating(completions, **kwargs):
    """Penalize use of external imports."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)

        if function is not None:
            ok, info = check_python_modules(function)
            scores.append(1.0 if ok else -20.0)  # 作弊重罚
        else:
            scores.append(-1.0)  # 创建函数失败

    return scores

**奖励3：策略成功**

奖励成功解决数独难题的策略。

In [ ]:
import numpy as np

global PRINTER
PRINTER = 0

def strategy_succeeds(completions, **kwargs):
    """Reward valid moves even if strategy eventually fails."""
    global PRINTER
    scores = []

    seed = np.random.randint(10000)
    difficulty = 40
    for completion in completions:
        printed = False
        response = completion[0]["content"]
        function = extract_function(response)

        if PRINTER % 5 == 0:
            printed = True
            print("\n" + "=" * 60)
            print(function)
            print("=" * 60)
        PRINTER += 1

        if function is not None:
            ok, info = check_python_modules(function)

        if function is None or "error" in info:
            scores.append(0)
            continue

        try:
            new_strategy = create_locked_down_function(function)
        except:
            scores.append(0)
            continue

        try:
            game = SudokuGame(difficulty = difficulty, seed = seed)
            valid_moves, game_state = execute_strategy(new_strategy, game)
            if valid_moves == difficulty:
                game_state = "success"

            print(f"\n Valid moves: {valid_moves}, Final state: {game_state}")

            if not printed:
                print("战略：")
                print(function[:200] + "..." if len(function) > 200 else function)

            print("\n最终板：")
            print(game.pretty())

            if game_state == "success":
                scores.append(30.0)  # 解开了谜题！
            elif valid_moves > 0:
                # 基于失败前所做的有效动作的奖励
                # 每个有效的移动值 0.2 分
                reward = valid_moves * 0.2
                scores.append(reward)
            else:
                scores.append(-2.0)  # 立即失败，没有有效的动作

        except TimeoutError:
            print("暂停")
            scores.append(-1.0)
        except Exception as e:
            print(f"Exception: {str(e)[:100]}")
            scores.append(-3.0)

    return scores

# 数据集准备

创建训练数据集。

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": prompt.strip()}],
        "answer": 0,
    }
] * 1000)

maximum_length = len(tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt.strip()}],
    add_generation_prompt = True
))

print(f"Maximum prompt length: {maximum_length}")
print("\n样本数据集：")
print(dataset[0])

Maximum prompt length: 830

Dataset sample:
{'prompt': [{'content': 'Create a Sudoku solving strategy using only native Python built-in functions without any import statements.\nYou are given two lists of lists (9x9 grids):\n- board: current state (0 means empty)\n- initial: starting puzzle (0 means was empty, numbers are fixed)\n\nReturn a tuple (row, col, number) for the next move.\n- row: 0-8 (row index)\n- col: 0-8 (column index)\n- number: 1-9 (digit to place)\n\nOnly place numbers in cells that are BOTH empty in initial AND empty in board (initial[row][col] == 0 AND board[row][col] == 0)\nUse Sudoku rules: no duplicates in rows, columns, or 3x3 boxes.\nOutput your function in backticks:\n```python\ndef strategy(board, initial):\n    # Your logic here\n    return (row, col, number)\n```\nAll helper functions must be inside def strategy. Output only the function.', 'role': 'user'}], 'answer': 0}


<a name="火车"></a>
### 训练模型

现在设置 GRPO Trainer 和所有配置！我们还支持 GSPO、GAPO、GRPO 博士等！前往 Unsloth [Reinforcement Learning Docs](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 了解更多选项。

In [ ]:
# 为提示留出空间（加上 1 个代币安全裕度）
max_completion_length = max_seq_length - (maximum_length + 1)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2, # 增加到 4 以使训练更顺畅
    num_generations = 2, # 如果内存不足则减少
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # 设置为 1 以进行完整的训练
    max_steps = 60,
    save_steps = 100,
    report_to = "none", # 可以使用权重和偏差、TrackIO
    output_dir = "outputs",
    epsilon = 0.2,
    epsilon_high = 0.28, # 一面
    delta = 1.5, # 两侧
    loss_type = 'bnpo',
    mask_truncated_completions = True
    # 用于可选培训+评估
    # fp16_full_eval = 真，
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "步骤",
    # 评估步骤 = 1,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


让我们运行训练器吧！如果向上滚动，您会看到奖励表。目标是看到“奖励”栏增加！

您可能需要等待 150 到 200 步才能执行任何操作。前 100 步您可能会获得较低的奖励。请耐心等待！

|步骤|训练损失|奖励 |奖励标准 |完成长度 |吉隆坡 |
|------|-------------|------------|------------|--------------------|----------|
| 1 | 0.000000 | 0.000000 0.125000 | 0.000000 | 0.000000 200.000000 | 0.000000 | 0.000000
| 2 | 0.000000 | 0.000000 0.072375 | 0.072375 0.248112 | 200.000000 | 0.000000 | 0.000000
| 3 | 0.000000 | 0.000000 -0.079000 | -0.079000 0.163776 | 182.500000 | 0.000005 | 0.000005

In [ ]:
# 用于可选培训+评估
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        function_works,
        no_cheating,
        strategy_succeeds,
    ],
    args = training_args,
    train_dataset = dataset,

    # 用于可选培训+评估
    # train_dataset = new_dataset["火车"],
    # eval_dataset = new_dataset["测试"],
)

让我们训练模型吧！

**注意** 遗憾的是，T4 免费 GPU 一代可能需要 5 分钟，因为它是旧 GPU - A100 或 H100 会快得多！

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 59,719,680 of 5,164,017,184 (1.16% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!

def strategy(board, initial):
    def get_candidates(r, c):
        if board[r][c] != 0:
            return set()

        used = set()

        # Check row
        for col in range(9):
            if board[r][col] != 0:
                used.add(board[r][col])

        # Check column
        for row in range(9):
            if board[row][c] != 0:
                used.add(board[row][c])

        # Check 3x3 box
        start_row = (r // 3) * 3
        start_col = (c // 3) * 3
        for i in range(3):
            for j in range(3):
                current_r = start_row + i
                current_c = start_col + j
                if board[current_r][current_c] != 0:
                    used.add(board[current_r][current_c])

        all_digits = set([1, 2, 3, 4, 5, 6, 7, 8, 9])
        return all_digits - used

    # 1. Find all empty cells where a move can potentially be made
    empty_cells = []
    for r in range(9):
        for 

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / function_works / mean,rewards / function_works / std,rewards / no_cheating / mean,rewards / no_cheating / std,rewards / strategy_succeeds / mean,rewards / strategy_succeeds / std
1,0.000000,20.600000,16.122034,501.000000,368.000000,634.000000,0.000000,501.000000,368.000000,634.000000,0.000006,1.000000,0.000000,1.000000,0.000000,18.600000,16.122034
2,0.000000,8.800000,0.000000,392.000000,380.000000,404.000000,0.000000,392.000000,380.000000,404.000000,0.000004,1.000000,0.000000,1.000000,0.000000,6.800000,0.000000
3,0.000000,8.400000,0.000000,338.500000,329.000000,348.000000,0.000000,338.500000,329.000000,348.000000,0.000042,1.000000,0.000000,1.000000,0.000000,6.400000,0.000000
4,0.000000,8.000000,0.000000,369.500000,360.000000,379.000000,0.000000,369.500000,360.000000,379.000000,0.000371,1.000000,0.000000,1.000000,0.000000,6.000000,0.000000
5,0.000001,5.800000,3.676955,470.000000,429.000000,511.000000,0.000000,470.000000,429.000000,511.000000,0.001449,1.000000,0.000000,1.000000,0.000000,3.800000,3.676955
6,0.000002,20.400000,16.404879,576.500000,415.000000,738.000000,0.000000,576.500000,415.000000,738.000000,0.002273,1.000000,0.000000,1.000000,0.000000,18.400000,16.404877
7,0.000031,7.500000,1.555635,404.500000,349.000000,460.000000,0.000000,404.500000,349.000000,460.000000,0.030822,1.000000,0.000000,1.000000,0.000000,5.500000,1.555635
8,0.000007,20.600000,16.122034,906.000000,811.000000,1001.000000,0.000000,906.000000,811.000000,1001.000000,0.007096,1.000000,0.000000,1.000000,0.000000,18.600000,16.122034
9,0.000005,8.900000,0.424264,737.500000,634.000000,841.000000,0.000000,737.500000,634.000000,841.000000,0.006361,1.000000,0.000000,1.000000,0.000000,6.900000,0.424264
10,0.000015,8.700001,0.141421,649.500000,631.000000,668.000000,0.000000,649.500000,631.000000,668.000000,0.012028,1.000000,0.000000,1.000000,0.000000,6.700000,0.141422



 Valid moves: 34, Final state: failed
Strategy:
def strategy(board, initial):
    def is_valid(b, r, c, num):
        # Check row
        for col in range(9):
            if col != c and b[r][col] == num:
                return False

        # Che...

Final board:
┌───────┬───────┬───────┐
│ 7 2 8 │ 3 1 4 │ 5 6 9 │
│ 5 3 1 │ 6 2 7 │ 4 8 . │
│ 6 . 4 │ 5 8 9 │ 3 2 7 │
├───────┼───────┼───────┤
│ 2 4 7 │ 1 6 3 │ 8 9 5 │
│ 3 5 9 │ 4 7 2 │ 6 1 . │
│ 1 8 6 │ . 9 5 │ 7 4 2 │
├───────┼───────┼───────┤
│ 4 7 2 │ 9 5 6 │ 1 3 8 │
│ 9 1 3 │ 8 4 . │ 2 7 6 │
│ 8 6 . │ 2 3 1 │ 9 5 4 │
└───────┴───────┴───────┘

 Valid moves: 34, Final state: failed
Strategy:
def strategy(board, initial):
    rows = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    cols = [0, 1, 2, 3, 4, 5, 6, 7, 8]

    def is_safe(b, r, c, num):
        # Check row
        for j in range(9):
            i...

Final board:
┌───────┬───────┬───────┐
│ 7 2 8 │ 3 1 4 │ 5 6 9 │
│ 5 3 1 │ 6 2 7 │ 4 8 . │
│ 6 . 4 │ 5 8 9 │ 3 2 7 │
├───────┼───────┼───────


 Valid moves: 38, Final state: failed
Strategy:
def strategy(board, initial):
    N = 9

    def is_valid_placement(r, c, num):
        # 1. Check Row
        for j in range(N):
            if j != c and board[r][j] == num:
                return F...

Final board:
┌───────┬───────┬───────┐
│ 5 3 9 │ 6 1 2 │ 4 7 8 │
│ 1 2 8 │ 3 7 4 │ 5 9 6 │
│ 4 7 6 │ 8 9 5 │ 1 2 3 │
├───────┼───────┼───────┤
│ 2 1 3 │ 4 5 7 │ 6 8 9 │
│ 6 8 5 │ 1 3 9 │ 7 4 2 │
│ 9 4 7 │ 2 6 8 │ 3 1 5 │
├───────┼───────┼───────┤
│ 7 6 4 │ 5 2 3 │ 8 . 1 │
│ 8 5 1 │ 9 4 6 │ 2 3 7 │
│ 3 9 2 │ 7 8 1 │ . 5 4 │
└───────┴───────┴───────┘

 Valid moves: 38, Final state: failed
Strategy:
def strategy(board, initial):
    N = 9

    def is_valid_placement(r, c, num):
        # 1. Check row
        for col in range(N):
            if col != c and board[r][col] == num:
                re...

Final board:
┌───────┬───────┬───────┐
│ 5 3 9 │ 6 1 2 │ 4 7 8 │
│ 1 2 8 │ 3 7 4 │ 5 9 6 │
│ 4 7 6 │ 8 9 5 │ 1 2 3 │
├───────┼───────┼───────

现在我们刚刚用 GRPO 训练了 LoRA - 我们首先保存 LoRA！

In [ ]:
model.save_pretrained("gemma_4_lora")  # 本地储蓄
tokenizer.save_pretrained("gemma_4_lora")

验证 LoRA 确实经过训练！

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # 验证 A 和 B 均非零
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

<a name="推理"></a>
# 推论
现在让我们尝试一下我们刚刚训练的模型！

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt.strip()}],
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer

_ = model.generate(
    **tokenizer(images = None,text = text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 512,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<a name="保存"></a>
### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 合并到16位
if False: model.save_pretrained_merged("gemma_4_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/gemma_4_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False: model.save_pretrained_merged("gemma_4_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/gemma_4_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("gemma_4_lora")
    tokenizer.save_pretrained("gemma_4_lora")
if False:
    model.push_to_hub("HF_USERNAME/gemma_4_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/gemma_4_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

[**新**] 要微调并自动导出到 Ollama，请尝试我们的 [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# 保存到8位Q8_0
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer,)
# 记得去 https://huggingface.co/settings/tokens 获取令牌！
# 并将 hf 更改为您的用户名！
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False: model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/gemma_4_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# 保存到多个 GGUF 选项 - 如果您想要多个，速度会更快！
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/gemma_4_finetune", # 将 hf 更改为您的用户名！
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

现在，使用 llama.cpp 中的 `gemma_4_finetune.Q8_0.gguf` 文件或 `gemma_4_finetune.Q4_K_M.gguf` 文件。

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1.训练自己的推理模型——Llama GRPO笔记本[Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. 将微调保存到Ollama。 [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 视觉微调 - 射线照相用例。 [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. 请参阅我们的 [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks) 上的 DPO、ORPO、持续预训练、对话微调等笔记本！

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️
</div>

  此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可。